# S&P 500 Market Cap Treemap

This notebook is self-contained.

It does all of this in one flow:

- downloads the current S&P 500 constituents from Wikipedia
- normalizes tickers for Yahoo compatibility
- fetches market capitalization with `yfinance`
- calculates each company's participation (`weight`)
- builds an interactive treemap similar to the project market-cap graphic
- saves both the cleaned source table and the HTML chart

Important note:

- this notebook uses the current S&P 500 constituents and current market caps
- it is not a historical point-in-time reconstruction of index membership


## Cell 1: Environment, paths, and outputs

This cell:

- detects the project root without absolute paths
- defines the output files for this notebook
- prepares the output directory


In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd
import plotly.express as px
import requests
import yfinance as yf
from bs4 import BeautifulSoup

cwd = Path.cwd().resolve()

def looks_like_project_root(path: Path) -> bool:
    return (path / 'data').exists() and (path / 'notebooks').exists()

candidates = [cwd, *cwd.parents]
ROOT = next((path for path in candidates if looks_like_project_root(path)), cwd)

if ROOT == cwd and not looks_like_project_root(ROOT):
    print(f'Warning: project root not detected. Using current working directory: {ROOT}')

OUT_DIR = ROOT / 'sources' / 'manual' / 'notebook'
CONSTITUENTS_OUT = OUT_DIR / 'sp500_current_constituents.csv'
TREEMAP_CSV = OUT_DIR / 'sp500_market_cap_treemap.csv'
TREEMAP_HTML = OUT_DIR / 'sp500_market_cap_treemap.html'
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('OUT_DIR =', OUT_DIR)
print('TREEMAP_CSV =', TREEMAP_CSV)
print('TREEMAP_HTML =', TREEMAP_HTML)


## Cell 2: Download the current S&P 500 constituents from Wikipedia

This cell:

- downloads the current constituents table from Wikipedia
- extracts `ticker`, `name`, `sector`, and `industry`
- normalizes tickers such as `BRK.B -> BRK-B`
- saves the current constituent list as a CSV snapshot


In [ ]:
def normalize_ticker(symbol: str) -> str:
    return symbol.strip().replace('.', '-')


resp = requests.get(
    WIKI_URL,
    headers={'User-Agent': 'Mozilla/5.0'},
    timeout=30,
)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, 'html.parser')
table = soup.select_one('#constituents')
if table is None:
    raise ValueError('Could not find Wikipedia constituents table')

rows = []
for tr in table.select('tbody tr'):
    tds = [td.get_text(' ', strip=True) for td in tr.select('td')]
    if len(tds) < 4:
        continue

    rows.append({
        'ticker': normalize_ticker(tds[0]),
        'name': tds[1],
        'sector': tds[2],
        'industry': tds[3],
    })

constituents_df = pd.DataFrame(rows)
constituents_df.to_csv(CONSTITUENTS_OUT, index=False)
constituents_df.head()


## Cell 3: Download market caps from Yahoo Finance and calculate participation

This cell:

- loops through the current constituent tickers
- pulls `market_cap` with `yfinance`
- falls back to `info['marketCap']` if `fast_info` is missing
- merges the result with the Wikipedia classification
- calculates `weight = marketCap / totalMarketCap * 100`

The result is the clean source table that drives the treemap.


In [ ]:
tickers = (
    constituents_df['ticker']
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

market_caps = []
for ticker in tickers:
    market_cap = None

    try:
        instrument = yf.Ticker(ticker)

        try:
            market_cap = instrument.fast_info.get('market_cap')
        except Exception:
            market_cap = None

        if market_cap is None:
            try:
                info = instrument.info
                market_cap = info.get('marketCap')
            except Exception:
                market_cap = None
    except Exception:
        market_cap = None

    market_caps.append({
        'ticker': ticker,
        'marketCap': market_cap,
    })

    time.sleep(0.2)

caps_df = pd.DataFrame(market_caps)
df = constituents_df.merge(caps_df, on='ticker', how='left')
df['marketCap'] = pd.to_numeric(df['marketCap'], errors='coerce')
df = df[df['marketCap'].notna() & (df['marketCap'] > 0)].copy()

total_market_cap = df['marketCap'].sum()
df['weight'] = df['marketCap'] / total_market_cap * 100
df = df.sort_values('weight', ascending=False).reset_index(drop=True)

df[['ticker', 'name', 'sector', 'industry', 'marketCap', 'weight']].head(10)


## Cell 4: Build the treemap

This cell:

- assigns sector colors
- uses `weight` as the box size
- uses `sector -> industry -> ticker` as the hierarchy
- writes a hover template with company, weight, sector, and industry


In [ ]:
SECTOR_COLORS = {
    'Information Technology': '#78b9ff',
    'Financials': '#22c55e',
    'Communication Services': '#b793ff',
    'Consumer Discretionary': '#8395ff',
    'Health Care': '#e7e3ff',
    'Industrials': '#ff9cbc',
    'Consumer Staples': '#ff985f',
    'Energy': '#f4c84d',
    'Utilities': '#d8df53',
    'Materials': '#ff6767',
    'Real Estate': '#b7ece0',
}

fig = px.treemap(
    df,
    path=['sector', 'industry', 'ticker'],
    values='weight',
    color='sector',
    color_discrete_map=SECTOR_COLORS,
    custom_data=['name', 'weight', 'sector', 'industry', 'marketCap'],
)

fig.update_traces(
    texttemplate='%{label}<br>%{value:.2f}%',
    hovertemplate=(
        '<b>%{label}</b><br>'
        'Company: %{customdata[0]}<br>'
        'Weight: %{customdata[1]:.2f}%<br>'
        'Sector: %{customdata[2]}<br>'
        'Industry: %{customdata[3]}<br>'
        'Market Cap: %{customdata[4]:,.0f}<extra></extra>'
    ),
    marker_line_width=1,
)

fig.update_layout(
    title='S&P 500 Market Cap Treemap',
    paper_bgcolor='#0a0908',
    plot_bgcolor='#0a0908',
    font=dict(color='white', size=14),
    margin=dict(t=70, l=10, r=10, b=10),
)

fig


## Cell 5: Save outputs

This cell writes:

- the cleaned source table used by the treemap
- the interactive HTML chart

That way the notebook both computes the data and leaves a reusable artifact behind.


In [ ]:
df.to_csv(TREEMAP_CSV, index=False)
fig.write_html(TREEMAP_HTML, include_plotlyjs='cdn')

print('Saved:')
print('-', CONSTITUENTS_OUT)
print('-', TREEMAP_CSV)
print('-', TREEMAP_HTML)
